In [1]:
### Global Imports
import os
import pandas as pd

In [2]:
### Import Checks

In [12]:
### Global Variables
DATA_SAVE_PATH = "../../models/used_data"
DAYS_PER_EPISODE = 3
TRAIN_TEST_SPLIT_FRACTION = 0.2 # Amount of data you reserve for testing.
BUFFER_DAYS = 3 # Amount of days that needs to be between the test episodes.

In [13]:
from Simulation.suite_simple_trading.data_splitting import get_or_create_train_test_split
from Simulation.suite_simple_trading.pre_processing import clean_data

### Getting data

raw_data_path = '../../data/2025_minute.csv'
cleaned_data_cache_path = '../../data/2025_minute_cleaned.pkl'

if os.path.exists(cleaned_data_cache_path):
    print(f"Loading cached cleaned data from: {cleaned_data_cache_path}")
    cleaned_df = pd.read_pickle(cleaned_data_cache_path)
else:
    print("No cached data found. Running the full cleaning process...")
    raw_df = pd.read_csv(raw_data_path, sep=';')
    cleaned_df = clean_data(raw_df)
    print(f"Saving cleaned data to cache: {cleaned_data_cache_path}")
    cleaned_df.to_pickle(cleaned_data_cache_path)

all_data = cleaned_df[['Datetime', 'Imbalance Price']]

train_df, test_df, nr_of_episodes = get_or_create_train_test_split(
        all_data=all_data,
        save_path=DATA_SAVE_PATH,
        days_per_episode=DAYS_PER_EPISODE,
        test_fraction=TRAIN_TEST_SPLIT_FRACTION,
        buffer_days=BUFFER_DAYS
    )

Loading cached cleaned data from: ../../data/2025_minute_cleaned.pkl
--- Loading existing train/test data from '../../models/used_data' ---
--- Data loaded successfully. Found 19 test episodes. ---
Test Data Size / Train Data Size: 0.25


In [1]:
from Simulation.suite_simple_trading.model import ExtendedBatteryEnv
from Simulation.suite_simple_trading.agent_trainer import train_ppo_agent

train_env = ExtendedBatteryEnv(
    battery_capacity_mwh=10.0,
    charge_discharge_rate_mw=5.0,
    all_data=train_df,
    days_per_episode=DAYS_PER_EPISODE
)
### Training
train_ppo_agent(
    env=train_env,
    model_save_path='models/ppo_battery_trading_model',
    reward_save_path=REWARD_SAVE_PATH,
    total_timesteps=100_000,
)

In [ ]:
### Testing